In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Configuration
TICKERS = ['NVDA', 'MU', 'AAPL', 'AMD', 'ASML', 'MSFT', 'GOOG']
MACRO_SYMBOLS = {
    'vix': '^VIX', 
    'dxy': 'DX-Y.NYB', 
    'oil': 'CL=F', 
    'tnx': '^TNX', # 10 Year Treasury Yield
    'irx': '^IRX'  # 13 Week Treasury Bill (Risk Free Rate Proxy)
}
START_DATE = '2014-01-01' 
END_DATE = '2025-05-01'

# -------------------------------------------------------
# 2. Data Download & MultiIndex Handling
# -------------------------------------------------------
print("Downloading Ticker Data...")
data = yf.download(TICKERS, start=START_DATE, end=END_DATE, interval='1wk', auto_adjust=False)

# Robust MultiIndex Handling
if isinstance(data.columns, pd.MultiIndex):
    # Extract specifically the Adj Close and Volume levels
    # level=0 is usually the Price Type ('Adj Close'), level=1 is Ticker
    try:
        adj_close = data.xs('Adj Close', axis=1, level=0)
        volume = data.xs('Volume', axis=1, level=0)
    except KeyError:
        # Fallback if yfinance changes level order
        adj_close = data.xs('Adj Close', axis=1, level=1)
        volume = data.xs('Volume', axis=1, level=1)
else:
    # Fallback for single level (unlikely with multiple tickers but safe to have)
    adj_close = data['Adj Close']
    volume = data['Volume']

adj_close = adj_close.ffill().dropna()
volume = volume.ffill().dropna()

print("Downloading Macro Data...")
macro_raw = yf.download(list(MACRO_SYMBOLS.values()), start=START_DATE, end=END_DATE, interval='1wk', auto_adjust=False)

# Macro MultiIndex Handling
if isinstance(macro_raw.columns, pd.MultiIndex):
    try:
        macro_data = macro_raw.xs('Adj Close', axis=1, level=0)
    except KeyError:
        macro_data = macro_raw.xs('Adj Close', axis=1, level=1)
else:
    macro_data = macro_raw['Adj Close']

# Rename Macro columns from symbols (^VIX) to friendly names (vix)
inv_map = {v: k for k, v in MACRO_SYMBOLS.items()}
macro_data = macro_data.rename(columns=inv_map)
macro_data = macro_data.ffill().dropna()

# -------------------------------------------------------
# 3. Helper Functions
# -------------------------------------------------------

def calculate_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_macd(series, fast=12, slow=26, signal=9):
    exp1 = series.ewm(span=fast, adjust=False).mean()
    exp2 = series.ewm(span=slow, adjust=False).mean()
    macd = exp1 - exp2
    signal_line = macd.ewm(span=signal, adjust=False).mean()
    histogram = macd - signal_line
    return histogram

def calculate_bollinger(series, window=20):
    sma = series.rolling(window=window).mean()
    std = series.rolling(window=window).std()
    # Add 1e-8 to avoid division by zero in flat markets
    width = ((sma + (std * 2)) - (sma - (std * 2))) / (sma + 1e-8)
    position = (series - (sma - (std * 2))) / ((sma + (std * 2)) - (sma - (std * 2)) + 1e-8)
    return width, position

def rolling_zscore(series, window=52):
    # Robust scaling: (Value - RollingMean) / RollingStd
    roll_mean = series.rolling(window=window).mean()
    roll_std = series.rolling(window=window).std()
    # Handle division by zero if std is 0 (flat price)
    z = (series - roll_mean) / (roll_std + 1e-8)
    return z

# -------------------------------------------------------
# 4. Feature Engineering Loop
# -------------------------------------------------------
print("Generating Features...")

feature_dict = {}

# --- A. Macro Features ---
for name in MACRO_SYMBOLS.keys():
    if name in macro_data.columns:
        feature_dict[f'{name}_norm'] = rolling_zscore(macro_data[name], window=52)

# Yield Curve (10Y - 3M approximation)
# Ensure both columns exist
if 'tnx' in macro_data.columns and 'irx' in macro_data.columns:
    feature_dict['yield_curve_norm'] = rolling_zscore(macro_data['tnx'] - macro_data['irx'], window=52)

# --- B. Ticker Specific Features ---
# First, Calculate Log Returns (Additivity property)
log_returns = np.log(adj_close / adj_close.shift(1))

for ticker in TICKERS:
    if ticker not in adj_close.columns:
        continue
        
    price = adj_close[ticker]
    vol = volume[ticker]
    
    # 1. Momentum
    # Note: We calculate these raw, then normalize them in Step D
    feature_dict[f'{ticker}_mom_1w'] = log_returns[ticker] 
    feature_dict[f'{ticker}_mom_4w'] = price.pct_change(4)
    feature_dict[f'{ticker}_mom_13w'] = price.pct_change(13)
    
    # 2. Volatility
    feature_dict[f'{ticker}_vol_4w'] = log_returns[ticker].rolling(4).std()
    feature_dict[f'{ticker}_vol_52w'] = log_returns[ticker].rolling(52).std()
    
    # 3. Price to Averages
    sma_12 = price.rolling(12).mean()
    ema_26 = price.ewm(span=26, adjust=False).mean()
    feature_dict[f'{ticker}_price_to_sma'] = (price / (sma_12 + 1e-8)) - 1
    feature_dict[f'{ticker}_price_to_ema'] = (price / (ema_26 + 1e-8)) - 1
    
    # 4. Technicals
    feature_dict[f'{ticker}_macd'] = calculate_macd(price)
    bb_w, bb_p = calculate_bollinger(price)
    feature_dict[f'{ticker}_bb_width'] = bb_w
    feature_dict[f'{ticker}_bb_pos'] = bb_p
    feature_dict[f'{ticker}_rsi'] = calculate_rsi(price) / 100.0 # RSI is 0-1 bounded
    
    # 5. Volume
    # Rolling sum of (Volume * Sign of Price Change)
    feature_dict[f'{ticker}_obv_roc'] = (vol * np.sign(price.diff())).rolling(4).sum()
    feature_dict[f'{ticker}_vol_roc'] = vol.pct_change(13)
    
    # 6. Drawdown
    roll_max = price.rolling(52, min_periods=1).max()
    feature_dict[f'{ticker}_dd'] = (price / (roll_max + 1e-8)) - 1.0

# --- C. Assemble DataFrame ---
df_features = pd.DataFrame(feature_dict)

# --- D. Normalization Step ---
# Apply Rolling Z-Score to everything except RSI (0-1) and already normalized Macro columns
# This explicitly INCLUDES 'mom_1w' (log returns) as requested
cols_to_normalize = [c for c in df_features.columns if 'rsi' not in c and 'norm' not in c]

for col in cols_to_normalize:
    # This creates, for example, 'NVDA_mom_1w_norm'
    df_features[f'{col}_norm'] = rolling_zscore(df_features[col], window=52)

# Keep only Normalized columns + RSI
final_cols = [c for c in df_features.columns if '_norm' in c or 'rsi' in c]
df_final = df_features[final_cols].copy()

# Add Calendar Features
df_final['month_sin'] = np.sin(2 * np.pi * df_final.index.month / 12)
df_final['month_cos'] = np.cos(2 * np.pi * df_final.index.month / 12)

# Add Raw Log Returns for the RL Environment (P&L Calculation)
# These are NOT input to the neural network, they are for the Environment step()
for ticker in TICKERS:
    df_final[f'{ticker}_log_ret'] = log_returns[ticker]

# Drop NaN (Rolling windows create NaNs at the start)
df_final = df_final.replace([np.inf, -np.inf], np.nan).dropna()

print(f"Final Dataset Shape: {df_final.shape}")
print(f"Date Range: {df_final.index.min()} to {df_final.index.max()}")
print("Columns preview:", df_final.columns[:5].tolist())

df_final.round(4).to_csv('advanced_portfolio_data.csv')
print("Saved to 'advanced_portfolio_data.csv'")

[*********************100%***********************]  7 of 7 completed
[*********************100%***********************]  5 of 5 completed

Generating Features...
Final Dataset Shape: (489, 113)
Date Range: 2015-12-23 00:00:00 to 2025-04-30 00:00:00
Columns preview: ['vix_norm', 'dxy_norm', 'oil_norm', 'tnx_norm', 'irx_norm']
Saved to 'advanced_portfolio_data.csv'


In [4]:
# 7. Download QQQ Benchmark (Monthly, expanded to weekly)
print("Downloading QQQ benchmark data...")
qqq_data = yf.download('QQQ', start=START_DATE, end=END_DATE, interval='1mo', auto_adjust=True, back_adjust=True)

# Handle MultiIndex columns from yfinance
if isinstance(qqq_data.columns, pd.MultiIndex):
    try:
        qqq_close = qqq_data.xs('Close', level=0, axis=1)
    except KeyError:
        qqq_close = qqq_data.xs('Adj Close', level=0, axis=1)
else:
    qqq_close = qqq_data['Close'] if 'Close' in qqq_data.columns else qqq_data['Adj Close']

# Calculate monthly returns
qqq_returns_monthly = qqq_close.pct_change().dropna()
qqq_returns_monthly.columns = ['QQQ']

# Expand monthly returns to weekly dates (forward-fill monthly value for all weeks in that month)
# This aligns QQQ with the weekly portfolio_data.csv dates
qqq_benchmark = qqq_returns_monthly.reindex(df_final.index, method='ffill').dropna()

# Save QQQ benchmark with weekly dates aligned to portfolio data
qqq_benchmark.round(4).to_csv('qqq_benchmark.csv')
print(f"QQQ benchmark saved: {len(qqq_benchmark)} weekly periods (monthly values forward-filled)")
print(f"QQQ benchmark date range: {qqq_benchmark.index[0]} to {qqq_benchmark.index[-1]}")
print(f"Aligned with portfolio data: {len(qqq_benchmark)} == {len(df_final)} periods")

[*********************100%***********************]  1 of 1 completed

QQQ benchmark saved: 489 weekly periods (monthly values forward-filled)
QQQ benchmark date range: 2015-12-23 00:00:00 to 2025-04-30 00:00:00
Aligned with portfolio data: 489 == 489 periods
